# Structured output (LangChain 1.3+)

Ask the model to return data matching a **schema** (not free-form text). LangChain parses the response into Python objects you can use directly.

**Why use it:** reliable JSON-like fields, validation, downstream code (`movie.year`, `contact.email`).

**Core API:** `model.with_structured_output(Schema)` → returns a **bound model** whose `.invoke()` yields a parsed object (not a raw `AIMessage`).

**Prerequisite:** `GROQ_API_KEY` in repo-root `.env`. See [2-modelintegration.ipynb](2-modelintegration.ipynb).

**Sections:** 1 Pydantic | 2 `include_raw` | 3 nested | 4 TypedDict | 5 dataclass | 6 `create_agent` | 7 when to use which

In [ ]:
"""Load API keys from .env."""
import os
from pathlib import Path

from dotenv import load_dotenv

for env_path in (Path.cwd() / ".env", Path.cwd().parent / ".env"):
    if env_path.is_file():
        load_dotenv(env_path)
        break
else:
    load_dotenv()

for key in ("GROQ_API_KEY", "OPENAI_API_KEY"):
    value = os.getenv(key)
    if value:
        os.environ[key] = value
    else:
        print(f"{key}: missing (skip sections that need it)")

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("groq:qwen/qwen3-32b")
model.profile.get("structured_output")  # True when provider supports it

## 1. Pydantic `BaseModel` (recommended)

**Pydantic** gives the richest schema: field types, `Field(description=...)`, validation, nested models.

1. Define a `BaseModel` with typed fields
2. `model.with_structured_output(Movie)` — sends schema to the API (often via tool / function calling)
3. `.invoke(prompt)` — returns a **`Movie` instance**, not an `AIMessage`

In [ ]:
from pydantic import BaseModel, Field


class Movie(BaseModel):
    """A movie with details."""

    title: str = Field(description="The title of the movie")
    year: int = Field(description="The year the movie was released")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The movie's rating out of 10")


model_with_structure = model.with_structured_output(Movie)
model_with_structure  # _ChatModelBinding + parser

### Plain `invoke` vs structured `invoke`

| | `model.invoke(...)` | `model_with_structure.invoke(...)` |
|--|---------------------|-------------------------------------|
| **Returns** | `AIMessage` (free text) | `Movie` (parsed object) |
| **Parsing** | You parse JSON yourself | LangChain handles it |
| **Use when** | Chat, prose | APIs, databases, typed pipelines |

In [ ]:
# Structured: returns Movie(title=..., year=..., ...)
movie = model_with_structure.invoke("Provide details about the movie 3 Idiots")
movie

In [ ]:
# Use like any Python object / Pydantic model
print(movie.title, movie.year)
print(movie.model_dump())  # dict for JSON APIs

## 2. Raw message + parsed object (`include_raw=True`)

Set `include_raw=True` when you need **both**:

- `parsed` — the `Movie` (or dict) object
- `raw` — the original `AIMessage` (token usage, `tool_calls`, reasoning metadata)

In [ ]:
model_with_raw = model.with_structured_output(Movie, include_raw=True)

result = model_with_raw.invoke("Provide details about the movie 3 Idiots")
result["parsed"]  # Movie instance
# result["raw"]   # AIMessage — uncomment to inspect metadata

## 3. Nested Pydantic structures

Use nested `BaseModel` classes and `list[...]` for arrays. Optional fields: `float | None = Field(None, ...)`.

In [ ]:
class Actor(BaseModel):
    name: str
    role: str


class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")


structured = model.with_structured_output(MovieDetails)
details = structured.invoke("Provide details about the movie 3 Idiots")
details

## 4. `TypedDict` (lighter, no runtime validation)

**TypedDict** uses stdlib `typing` — good when you want schema hints for the LLM but **not** Pydantic validation.

- Returns a **plain `dict`** (not a class instance)
- Use `Annotated[type, ..., "description"]` for field docs
- Do **not** use `pydantic.Field` on TypedDict (use `total=False` or `| None` for optionals)

In [ ]:
from typing import Annotated

from typing_extensions import TypedDict


class MovieDict(TypedDict):
    """A movie with details."""

    title: Annotated[str, "The title of the movie"]
    year: Annotated[int, "The year the movie was released"]
    director: Annotated[str, "The director of the movie"]
    rating: Annotated[float, "The movie's rating out of 10"]


model_typed = model.with_structured_output(MovieDict)
movie_dict = model_typed.invoke("Details for the movie 3 Idiots")
movie_dict  # dict with keys title, year, director, rating

In [ ]:
class ActorDict(TypedDict):
    name: str
    role: str


class MovieDetailsDict(TypedDict):
    title: str
    year: int
    cast: list[ActorDict]
    genres: list[str]
    budget: float | None


structured_td = model.with_structured_output(MovieDetailsDict)
structured_td.invoke("Provide details about the movie 3 Idiots")

## 5. `@dataclass`

A **dataclass** is a simple data container (stdlib). LangChain can use it as a schema similar to Pydantic, with less validation overhead.

In [ ]:
from dataclasses import dataclass


@dataclass
class MovieDC:
    title: str
    year: int
    director: str
    rating: float


model_dc = model.with_structured_output(MovieDC)
model_dc.invoke("Details for the movie 3 Idiots")

## 6. Structured output with `create_agent`

Pass `response_format=YourSchema` to `create_agent`. The agent adds a **ProviderStrategy** and puts the parsed result in `result["structured_response"]`.

Requires **OpenAI** here (`gpt-4o-mini`). Skip if `OPENAI_API_KEY` is missing.

In [ ]:
from langchain.agents import create_agent
from pydantic import BaseModel, Field


class ContactInfo(BaseModel):
    """Contact information for a person."""

    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")


agent = create_agent(
    model="gpt-4o-mini",
    response_format=ContactInfo,
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567",
            }
        ]
    }
)

result["structured_response"]  # ContactInfo instance

## 7. When to use which schema type?

| Schema | Returns | Validation | Best for |
|--------|---------|------------|----------|
| **Pydantic `BaseModel`** | Model instance | ✅ runtime | Production, nested data, agents |
| **`TypedDict`** | `dict` | ❌ type hints only | Quick prototypes, JSON-like output |
| **`@dataclass`** | Dataclass instance | ❌ minimal | Simple flat structs |

| API | When |
|-----|------|
| `model.with_structured_output(Schema)` | Single LLM call → typed result |
| `with_structured_output(..., include_raw=True)` | Need `AIMessage` metadata too |
| `create_agent(..., response_format=Schema)` | Agent loop + structured final answer |

```mermaid
%%{init: {"theme":"base","themeVariables":{"primaryColor":"#ffffff","primaryTextColor":"#111827","primaryBorderColor":"#374151","lineColor":"#374151","mainBkg":"#ffffff","nodeTextColor":"#111827"}}}%%
flowchart LR
  P[Prompt] --> M[Chat model]
  M -->|plain invoke| AI[AIMessage text]
  M -->|with_structured_output| SCH[Schema enforced]
  SCH --> OUT[Movie / dict / dataclass]
  style SCH fill:#eff6ff,stroke:#2563eb,color:#111827
  style OUT fill:#ecfdf5,stroke:#059669,color:#111827
```

### Summary

```python
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="...")
    year: int

structured = model.with_structured_output(Movie)
movie = structured.invoke("Tell me about Inception")  # -> Movie

both = model.with_structured_output(Movie, include_raw=True).invoke("...")
# both["parsed"], both["raw"]
```

**Related:** [3_tools.ipynb](3_tools.ipynb) (tool calling uses similar JSON schemas), [4_messages.ipynb](4_messages.ipynb) (message types).